# Customer Transaction Analytics with PySpark

**End-to-end Google Colab project**

Flow:

Customer Transactions → Large Dataset → PySpark → Data Cleaning → Sales Analysis → Customer Segmentation → Top Products → Business Insights

This notebook creates a synthetic dataset of 1 million transactions and performs the complete analysis using PySpark.

In [45]:
!pip install -q pyspark

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("CustomerTransactionAnalytics") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("PySpark started successfully")
print("Spark version:", spark.version)

PySpark started successfully
Spark version: 4.0.4


## 1. Create a Large Customer Transactions Dataset

We generate **1,000,000 transactions** for 50,000 customers.

In [47]:
random.seed(42)

NUM_RECORDS = 1_000_000
NUM_CUSTOMERS = 50_000

products = [
    ("Laptop", "Electronics", 60000),
    ("Smartphone", "Electronics", 30000),
    ("Headphones", "Electronics", 3000),
    ("Keyboard", "Electronics", 1500),
    ("Mouse", "Electronics", 800),
    ("Office Chair", "Furniture", 12000),
    ("Desk", "Furniture", 15000),
    ("Notebook", "Stationery", 150),
    ("Pen", "Stationery", 50),
    ("Backpack", "Accessories", 2000)
]

cities = [
    "Chennai", "Bangalore", "Mumbai",
    "Delhi", "Hyderabad", "Pune", "Coimbatore"
]

start_date = datetime(2025, 1, 1)

data = []

for i in range(NUM_RECORDS):
    customer_id = random.randint(1, NUM_CUSTOMERS)
    product, category, price = random.choice(products)
    quantity = random.randint(1, 5)

    transaction_date = start_date + timedelta(
        days=random.randint(0, 364)
    )

    data.append((
        i + 1,
        customer_id,
        transaction_date.strftime("%Y-%m-%d"),
        product,
        category,
        quantity,
        float(price),
        random.choice(cities)
    ))

print(f"Generated {len(data):,} transactions")

Generated 1,000,000 transactions


## 2. Create the PySpark DataFrame

In [48]:
schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("transaction_date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("city", StringType(), True)
])

df = spark.createDataFrame(data, schema)

print("Rows:", f"{df.count():,}")
print("Columns:", len(df.columns))

df.show(10, truncate=False)

Rows: 1,000,000
Columns: 8
+--------------+-----------+----------------+------------+-----------+--------+-------+----------+
|transaction_id|customer_id|transaction_date|product     |category   |quantity|price  |city      |
+--------------+-----------+----------------+------------+-----------+--------+-------+----------+
|1             |41906      |2025-05-21      |Smartphone  |Electronics|1       |30000.0|Bangalore |
|2             |14629      |2025-12-13      |Headphones  |Electronics|1       |3000.0 |Pune      |
|3             |35742      |2025-08-05      |Smartphone  |Electronics|5       |30000.0|Chennai   |
|4             |1953       |2025-04-30      |Smartphone  |Electronics|2       |30000.0|Hyderabad |
|5             |39454      |2025-04-12      |Laptop      |Electronics|5       |60000.0|Pune      |
|6             |42591      |2025-04-23      |Pen         |Stationery |4       |50.0   |Delhi     |
|7             |38619      |2025-03-23      |Mouse       |Electronics|1       |800

## 3. Inspect the Dataset

In [49]:
df.printSchema()

df.describe(
    "quantity", "price"
).show()

root
 |-- transaction_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- transaction_date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- city: string (nullable = true)

+-------+------------------+------------------+
|summary|          quantity|             price|
+-------+------------------+------------------+
|  count|           1000000|           1000000|
|   mean|          3.001048|        12484.1885|
| stddev|1.4138864525735002|18289.934026484145|
|    min|                 1|              50.0|
|    max|                 5|           60000.0|
+-------+------------------+------------------+



## 4. Data Cleaning

In [50]:
# Missing-value check
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

print("Missing values:")
missing_values.show()

# Remove duplicate transaction IDs
before_count = df.count()
df_clean = df.dropDuplicates(["transaction_id"])
after_count = df_clean.count()

print("Rows before duplicate removal:", f"{before_count:,}")
print("Rows after duplicate removal :", f"{after_count:,}")

# Convert date string to DateType
df_clean = df_clean.withColumn(
    "transaction_date",
    to_date("transaction_date", "yyyy-MM-dd")
)

# Remove invalid quantity/price records
df_clean = df_clean.filter(
    (col("quantity") > 0) &
    (col("price") > 0) &
    col("customer_id").isNotNull() &
    col("product").isNotNull()
)

# Create revenue
df_clean = df_clean.withColumn(
    "sales_amount",
    round(col("quantity") * col("price"), 2)
)

print("Final cleaned rows:", f"{df_clean.count():,}")

df_clean.show(10, truncate=False)

Missing values:
+--------------+-----------+----------------+-------+--------+--------+-----+----+
|transaction_id|customer_id|transaction_date|product|category|quantity|price|city|
+--------------+-----------+----------------+-------+--------+--------+-----+----+
|             0|          0|               0|      0|       0|       0|    0|   0|
+--------------+-----------+----------------+-------+--------+--------+-----+----+

Rows before duplicate removal: 1,000,000
Rows after duplicate removal : 1,000,000
Final cleaned rows: 1,000,000
+--------------+-----------+----------------+------------+-----------+--------+-------+---------+------------+
|transaction_id|customer_id|transaction_date|product     |category   |quantity|price  |city     |sales_amount|
+--------------+-----------+----------------+------------+-----------+--------+-------+---------+------------+
|1             |41906      |2025-05-21      |Smartphone  |Electronics|1       |30000.0|Bangalore|30000.0     |
|3          

## 5. Overall Sales Analysis

In [51]:
from pyspark.sql.functions import countDistinct, sum, count, avg

overall_sales = df_clean.agg(
    sum("sales_amount").alias("total_revenue"),
    sum("quantity").alias("total_units_sold"),
    count("transaction_id").alias("total_transactions"),
    countDistinct("customer_id").alias("unique_customers"),
    avg("sales_amount").alias("average_transaction_value")
)

overall_sales.show(truncate=False)

+--------------+----------------+------------------+----------------+-------------------------+
|total_revenue |total_units_sold|total_transactions|unique_customers|average_transaction_value|
+--------------+----------------+------------------+----------------+-------------------------+
|3.747855825E10|3001048         |1000000           |50000           |37478.55825              |
+--------------+----------------+------------------+----------------+-------------------------+



## 6. Sales by Category

In [52]:
category_sales = df_clean.groupBy("category").agg(
    round(sum("sales_amount"), 2).alias("total_sales"),
    sum("quantity").alias("units_sold"),
    count("transaction_id").alias("transactions")
).orderBy(desc("total_sales"))

category_sales.show(truncate=False)

+-----------+-------------+----------+------------+
|category   |total_sales  |units_sold|transactions|
+-----------+-------------+----------+------------+
|Electronics|2.87126619E10|1501112   |499944      |
|Furniture  |8.10234E9    |600140    |200166      |
|Accessories|6.0379E8     |301895    |100500      |
|Stationery |5.976635E7   |597901    |199390      |
+-----------+-------------+----------+------------+



## 7. Sales by City

In [53]:
city_sales = df_clean.groupBy("city").agg(
    round(sum("sales_amount"), 2).alias("total_sales"),
    sum("quantity").alias("units_sold"),
    count("transaction_id").alias("transactions")
).orderBy(desc("total_sales"))

city_sales.show(truncate=False)

+----------+------------+----------+------------+
|city      |total_sales |units_sold|transactions|
+----------+------------+----------+------------+
|Pune      |5.3771738E9 |429304    |143235      |
|Hyderabad |5.3628298E9 |428947    |142747      |
|Delhi     |5.35806035E9|426776    |142280      |
|Mumbai    |5.3557109E9 |430091    |143287      |
|Bangalore |5.35209035E9|430840    |143300      |
|Coimbatore|5.33753345E9|426653    |142296      |
|Chennai   |5.3351596E9 |428437    |142855      |
+----------+------------+----------+------------+



## 8. Monthly Sales Trend

In [54]:
monthly_sales = df_clean.withColumn(
    "month",
    date_format("transaction_date", "yyyy-MM")
).groupBy("month").agg(
    round(sum("sales_amount"), 2).alias("total_sales")
).orderBy("month")

monthly_sales.show(20, truncate=False)

+-------+------------+
|month  |total_sales |
+-------+------------+
|2025-01|3.18162545E9|
|2025-02|2.87700775E9|
|2025-03|3.1810673E9 |
|2025-04|3.0458899E9 |
|2025-05|3.20219235E9|
|2025-06|3.075908E9  |
|2025-07|3.16156035E9|
|2025-08|3.2118841E9 |
|2025-09|3.0998256E9 |
|2025-10|3.18747935E9|
|2025-11|3.0831892E9 |
|2025-12|3.1709289E9 |
+-------+------------+



## 9. Product Performance

In [55]:
product_sales = df_clean.groupBy("product", "category").agg(
    round(sum("sales_amount"), 2).alias("total_sales"),
    sum("quantity").alias("units_sold"),
    count("transaction_id").alias("transactions")
).orderBy(desc("total_sales"))

product_sales.show(20, truncate=False)

+------------+-----------+-----------+----------+------------+
|product     |category   |total_sales|units_sold|transactions|
+------------+-----------+-----------+----------+------------+
|Laptop      |Electronics|1.812138E10|302023    |100472      |
|Smartphone  |Electronics|9.00771E9  |300257    |100169      |
|Desk        |Furniture  |4.5033E9   |300220    |100200      |
|Office Chair|Furniture  |3.59904E9  |299920    |99966       |
|Headphones  |Electronics|8.92752E8  |297584    |99047       |
|Backpack    |Accessories|6.0379E8   |301895    |100500      |
|Keyboard    |Electronics|4.496175E8 |299745    |99896       |
|Mouse       |Electronics|2.412024E8 |301503    |100360      |
|Notebook    |Stationery |4.480695E7 |298713    |99640       |
|Pen         |Stationery |1.49594E7  |299188    |99750       |
+------------+-----------+-----------+----------+------------+



## 10. Top 10 Products

In [56]:
top_products = product_sales.limit(10)

top_products.show(10, truncate=False)

+------------+-----------+-----------+----------+------------+
|product     |category   |total_sales|units_sold|transactions|
+------------+-----------+-----------+----------+------------+
|Laptop      |Electronics|1.812138E10|302023    |100472      |
|Smartphone  |Electronics|9.00771E9  |300257    |100169      |
|Desk        |Furniture  |4.5033E9   |300220    |100200      |
|Office Chair|Furniture  |3.59904E9  |299920    |99966       |
|Headphones  |Electronics|8.92752E8  |297584    |99047       |
|Backpack    |Accessories|6.0379E8   |301895    |100500      |
|Keyboard    |Electronics|4.496175E8 |299745    |99896       |
|Mouse       |Electronics|2.412024E8 |301503    |100360      |
|Notebook    |Stationery |4.480695E7 |298713    |99640       |
|Pen         |Stationery |1.49594E7  |299188    |99750       |
+------------+-----------+-----------+----------+------------+



## 11. Customer Analysis

In [57]:
customer_summary = df_clean.groupBy("customer_id").agg(
    count("transaction_id").alias("transaction_count"),
    sum("quantity").alias("total_quantity"),
    round(sum("sales_amount"), 2).alias("total_spending"),
    round(avg("sales_amount"), 2).alias("average_transaction_value")
)

customer_summary.show(10)

+-----------+-----------------+--------------+--------------+-------------------------+
|customer_id|transaction_count|total_quantity|total_spending|average_transaction_value|
+-----------+-----------------+--------------+--------------+-------------------------+
|      35351|               24|            75|      903950.0|                 37664.58|
|      16861|               22|            57|     1104200.0|                 50190.91|
|       3997|               27|            79|     1583900.0|                 58662.96|
|      37489|               17|            57|      503500.0|                 29617.65|
|      11141|               15|            38|      582300.0|                  38820.0|
|       3918|               22|            64|     1092600.0|                 49663.64|
|       7240|               18|            61|      751750.0|                 41763.89|
|      43302|               17|            55|      528050.0|                 31061.76|
|      20382|               24| 

## 12. Customer Segmentation

Business rules:

- **Premium**: spending >= ₹100,000
- **High Value**: spending >= ₹50,000
- **Medium Value**: spending >= ₹20,000
- **Low Value**: spending < ₹20,000

In [58]:
customer_segment = customer_summary.withColumn(
    "segment",
    when(col("total_spending") >= 100000, "Premium")
    .when(col("total_spending") >= 50000, "High Value")
    .when(col("total_spending") >= 20000, "Medium Value")
    .otherwise("Low Value")
)

customer_segment.show(20)

+-----------+-----------------+--------------+--------------+-------------------------+-------+
|customer_id|transaction_count|total_quantity|total_spending|average_transaction_value|segment|
+-----------+-----------------+--------------+--------------+-------------------------+-------+
|      35351|               24|            75|      903950.0|                 37664.58|Premium|
|      16861|               22|            57|     1104200.0|                 50190.91|Premium|
|       3997|               27|            79|     1583900.0|                 58662.96|Premium|
|      37489|               17|            57|      503500.0|                 29617.65|Premium|
|      11141|               15|            38|      582300.0|                  38820.0|Premium|
|       3918|               22|            64|     1092600.0|                 49663.64|Premium|
|       7240|               18|            61|      751750.0|                 41763.89|Premium|
|      43302|               17|         

## 13. Segment Summary

In [59]:
segment_summary = customer_segment.groupBy("segment").agg(
    count("customer_id").alias("customer_count"),
    round(avg("total_spending"), 2).alias("average_spending"),
    round(sum("total_spending"), 2).alias("segment_revenue")
).orderBy(desc("segment_revenue"))

segment_summary.show(truncate=False)

+------------+--------------+----------------+---------------+
|segment     |customer_count|average_spending|segment_revenue|
+------------+--------------+----------------+---------------+
|Premium     |49864         |751423.91       |3.746900205E10 |
|High Value  |113           |76855.31        |8684650.0      |
|Medium Value|22            |38752.27        |852550.0       |
|Low Value   |1             |19000.0         |19000.0        |
+------------+--------------+----------------+---------------+



## 14. Top 10 Customers

In [60]:
top_customers = customer_segment.orderBy(
    desc("total_spending")
).limit(10)

top_customers.show(10, truncate=False)

+-----------+-----------------+--------------+--------------+-------------------------+-------+
|customer_id|transaction_count|total_quantity|total_spending|average_transaction_value|segment|
+-----------+-----------------+--------------+--------------+-------------------------+-------+
|13980      |29               |96            |2556150.0     |88143.1                  |Premium|
|4105       |27               |79            |2471300.0     |91529.63                 |Premium|
|47315      |32               |110           |2460850.0     |76901.56                 |Premium|
|344        |36               |105           |2379250.0     |66090.28                 |Premium|
|36445      |24               |76            |2341950.0     |97581.25                 |Premium|
|45403      |30               |116           |2280200.0     |76006.67                 |Premium|
|4031       |27               |95            |2276650.0     |84320.37                 |Premium|
|39527      |33               |107      

## 15. Business Insights

In [ ]:
overall = overall_sales.first()
best_category = category_sales.first()
best_city = city_sales.first()
best_product = product_sales.first()
best_customer = top_customers.first()

print("=" * 70)
print("BUSINESS INSIGHTS")
print("=" * 70)

print(f"1. Total revenue: ₹{overall['total_revenue']:,.2f}")
print(f"2. Total units sold: {overall['total_units_sold']:,}")
print(f"3. Total transactions: {overall['total_transactions']:,}")
print(f"4. Unique customers: {overall['unique_customers']:,}") # Commented out as 'unique_customers' is not present in 'overall_sales'
print(f"5. Average transaction value: ₹{overall['average_transaction_value']:,.2f}")

print(
    f"6. Best category: {best_category['category']} "
    f"(₹{best_category['total_sales']:,.2f})"
)

print(
    f"7. Best city: {best_city['city']} "
    f"(₹{best_city['total_sales']:,.2f})"
)

print(
    f"8. Best product: {best_product['product']} "
    f"(₹{best_product['total_sales']:,.2f})"
)

print(
    f"9. Highest-spending customer: "
    f"Customer {best_customer['customer_id']} "
    f"(₹{best_customer['total_spending']:,.2f})"
)

print("=" * 70)

## 16. Visualization - Sales by Category

In [ ]:
category_pd = category_sales.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(category_pd["category"], category_pd["total_sales"])
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 17. Visualization - Sales by City

In [ ]:
city_pd = city_sales.toPandas()

plt.figure(figsize=(10, 5))
plt.bar(city_pd["city"], city_pd["total_sales"])
plt.title("Sales by City")
plt.xlabel("City")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 18. Visualization - Monthly Sales

In [ ]:
monthly_pd = monthly_sales.toPandas()

plt.figure(figsize=(13, 5))
plt.plot(
    monthly_pd["month"],
    monthly_pd["total_sales"],
    marker="o"
)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 19. Visualization - Top Products

In [ ]:
top_product_pd = product_sales.limit(10).toPandas()

plt.figure(figsize=(10, 6))
plt.barh(
    top_product_pd["product"],
    top_product_pd["total_sales"]
)
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (₹)")
plt.ylabel("Product")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 20. Visualization - Customer Segments

In [ ]:
segment_pd = segment_summary.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(
    segment_pd["segment"],
    segment_pd["customer_count"]
)
plt.title("Customers by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Final Business Summary

This project demonstrates how PySpark can be used to process a large transaction dataset and convert raw customer transactions into actionable business insights.

### Key analysis performed

1. Created 1 million customer transactions
2. Loaded data into a PySpark DataFrame
3. Checked and cleaned the data
4. Calculated sales revenue
5. Analysed sales by category and city
6. Analysed monthly sales trends
7. Identified top products
8. Calculated customer spending
9. Segmented customers into value groups
10. Identified high-value customers
11. Generated business insights
12. Visualized the results

### Business questions answered

- How much revenue did the business generate?
- Which product category performs best?
- Which city generates the highest revenue?
- Which products are the top performers?
- Who are the highest-value customers?
- How many customers belong to each segment?
- How does revenue change month by month?

In [ ]:
# Stop Spark when the notebook is complete
spark.stop()
print("Spark session stopped successfully.")